# 01A — Petrobras 3W sector pack

**Outcome:** translate a deterministic development population of real-well 3W
recordings into the same Pack interface the Telecom pack uses. The default
keeps up to five recordings per real well/event pair and preserves a fixed
whole-well split; the actual denominators are reported rather than assumed.

Each source recording is one observation episode. `class` and `state` go to
`PACK-EVAL` and never to model input. The notebook invents no manifold
topology, tickets, maintenance events, severity or cross-well causes.

Same three steps: **describe the source**, **translate it**, **prove isolation**.

## 1. Setup

In [ ]:
import configparser
import hashlib
import os
import shutil
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (Path("/content/drive/MyDrive/anomaly_detection")
                     if IN_COLAB else Path.home() / "anomaly_detection_data")
DATA_ROOT = Path(os.getenv("ANOMALY_DATA_ROOT")
                 or os.getenv("ANOMALY_DRIVE_ROOT")
                 or default_data_root).expanduser()
default_code_root = (DATA_ROOT / "research" / "milestone1" if IN_COLAB
                     else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
                     else Path.cwd() / "notebooks" / "drive_research")
NOTEBOOK_HOME = Path(os.getenv("ANOMALY_NOTEBOOK_HOME", default_code_root)).expanduser()
if not (NOTEBOOK_HOME / "milestone1_core.py").is_file():
    raise FileNotFoundError(f"milestone1_core.py was not found in {NOTEBOOK_HOME}")
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    EVAL_SCHEMAS,
    PACK_SCHEMAS,
    SPLIT_SCHEMAS,
    fault_coverage,
    pack_fingerprint,
    read_json,
    save_pack,
    source_file,
    truth_like_columns,
)

SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    DATA_ROOT / "sources" / "petrobras_3w" / "2.0.0" / "raw" / "3w_dataset_2.0.0",
))
PACK_RUN_ID = os.getenv("THREEW_PACK_RUN_ID", "real_wells_expanded_v0_8_0")
PACK_ROOT = DATA_ROOT / "outputs" / "packs" / "petrobras_3w" / PACK_RUN_ID
FILES_PER_WELL_EVENT = int(os.getenv("THREEW_FILES_PER_WELL_EVENT", "5"))
SPLIT_SEED = int(os.getenv("THREEW_SPLIT_SEED", "0"))
BATCH_ROWS = int(os.getenv("THREEW_BATCH_ROWS", "50000"))
EXPECTED_FILES = int(os.getenv("THREEW_EXPECTED_FILES", "2228"))  # official 2.0.0 inventory
RUN_BUILD = os.getenv("RUN_THREEW_PACK", "1") == "1"

display(pd.Series({
    "runtime": "Colab + Drive" if IN_COLAB else "local Python",
    "data_root": str(DATA_ROOT),
    "code_root": str(NOTEBOOK_HOME),
    "source": str(SOURCE),
    "pack_root": str(PACK_ROOT),
    "files_per_well_event": FILES_PER_WELL_EVENT,
    "expected_inventory": EXPECTED_FILES,
    "whole_well_split_seed": SPLIT_SEED,
}, name="value").to_frame())

## 2. Petrobras 3W phrasebook

Units follow the official `dataset.ini`.

`sampling_mode="recording"` with a declared one-second cadence says: the
source samples at 1 Hz **inside** a file and promises nothing between files.
The adapter finds gaps within an `(entity, episode, metric)` series only, so
a hole inside a recording is reported while the interval between two
recordings is correctly left undefined.

In [ ]:
CADENCE_SECONDS = 1
METRIC_COLUMNS = ["native_field", *PACK_SCHEMAS["metric_catalogue"]]


def metric(native, metric_id, kind, unit):
    return (native, metric_id, "oil_well", kind, unit, "recording", CADENCE_SECONDS)


metric_map = pd.DataFrame([
    metric("ABER-CKGL",    "gas_lift_choke_opening",                     "gauge",          "percent"),
    metric("ABER-CKP",     "production_choke_opening",                   "gauge",          "percent"),
    metric("ESTADO-DHSV",  "downhole_safety_valve_state",                "discrete_state", "state_code"),
    metric("ESTADO-M1",    "production_master_valve_state",              "discrete_state", "state_code"),
    metric("ESTADO-M2",    "annulus_master_valve_state",                 "discrete_state", "state_code"),
    metric("ESTADO-PXO",   "pig_crossover_valve_state",                  "discrete_state", "state_code"),
    metric("ESTADO-SDV-GL", "gas_lift_shutdown_valve_state",             "discrete_state", "state_code"),
    metric("ESTADO-SDV-P", "production_shutdown_valve_state",            "discrete_state", "state_code"),
    metric("ESTADO-W1",    "production_wing_valve_state",                "discrete_state", "state_code"),
    metric("ESTADO-W2",    "annulus_wing_valve_state",                   "discrete_state", "state_code"),
    metric("ESTADO-XO",    "crossover_valve_state",                      "discrete_state", "state_code"),
    metric("P-ANULAR",     "annulus_pressure",                           "gauge",          "Pa"),
    metric("P-JUS-BS",     "service_pump_downstream_pressure",           "gauge",          "Pa"),
    metric("P-JUS-CKGL",   "gas_lift_choke_downstream_pressure",         "gauge",          "Pa"),
    metric("P-JUS-CKP",    "production_choke_downstream_pressure",       "gauge",          "Pa"),
    metric("P-MON-CKGL",   "gas_lift_choke_upstream_pressure",           "gauge",          "Pa"),
    metric("P-MON-CKP",    "production_choke_upstream_pressure",         "gauge",          "Pa"),
    metric("P-MON-SDV-P",  "production_shutdown_valve_upstream_pressure", "gauge",         "Pa"),
    metric("P-PDG",        "downhole_pressure",                          "gauge",          "Pa"),
    metric("PT-P",         "production_tube_downstream_pressure",        "gauge",          "Pa"),
    metric("P-TPT",        "tubing_pressure",                            "gauge",          "Pa"),
    metric("QBS",          "service_pump_flow_rate",                     "gauge",          "m3/s"),
    metric("QGL",          "gas_lift_flow_rate",                         "gauge",          "m3/s"),
    metric("T-JUS-CKP",    "production_choke_downstream_temperature",    "gauge",          "degC"),
    metric("T-MON-CKP",    "production_choke_upstream_temperature",      "gauge",          "degC"),
    metric("T-PDG",        "downhole_temperature",                       "gauge",          "degC"),
    metric("T-TPT",        "tubing_temperature",                         "gauge",          "degC"),
], columns=METRIC_COLUMNS)

EVALUATION_ONLY_FIELDS = ["class", "state"]

# Dataset.ini supplies the valve/choke rules. Three unmistakable repeated
# historian sentinels were verified across the real WELL files. Other
# finite extremes remain measured and are reviewed in EDA.
SOURCE_QUALITY_RULES = {
    metric_id: {"allowed_values": (0.0, 0.5, 1.0),
                "source": "dataset.ini valve-state values"}
    for metric_id in metric_map.loc[
        metric_map["measurement_kind"].eq("discrete_state"), "metric_id"
    ]
}
for metric_id in ("gas_lift_choke_opening", "production_choke_opening"):
    SOURCE_QUALITY_RULES[metric_id] = {
        "minimum": 0.0, "maximum": 100.0,
        "source": "dataset.ini choke opening [%]",
    }

SOURCE_QUALITY_RULES.update({
    "downhole_pressure": {
        "invalid_values": (-1.180116e42,),
        "source": "empirically verified repeated historian sentinel",
    },
    "downhole_temperature": {
        "invalid_values": (-1.711613e38, 30000.0),
        "source": "empirically verified repeated historian sentinels",
    },
})

assert len(metric_map) == 27
assert metric_map["metric_id"].is_unique
assert not truth_like_columns(metric_map["metric_id"])
assert set(EVALUATION_ONLY_FIELDS).isdisjoint(metric_map["native_field"])
display(metric_map)
display(pd.DataFrame.from_dict(SOURCE_QUALITY_RULES, orient="index")
        .rename_axis("metric_id").reset_index())

## 3. Inventory, development population and whole-well split

Duplicate download variants such as `file (1).parquet` are excluded and the
official inventory count is checked. Real recordings are those named `WELL-*`.

Selection is deterministic and never splits rows or timestamps: **entire
wells** go to calibration, development or holdout. A reproducible hash search
guarantees that development and holdout each contain normal and event
recordings. Missing event types are reported, not fabricated.

In [ ]:
def read_inventory(source):
    parser = configparser.ConfigParser(interpolation=None)
    if not parser.read(Path(source) / "dataset.ini", encoding="utf-8"):
        raise FileNotFoundError(f"Missing 3W metadata: {source}/dataset.ini")
    event_directories = sorted(
        path for path in (Path(source) / str(code) for code in range(10)) if path.is_dir()
    )
    all_files = [path for directory in event_directories for path in sorted(directory.glob("*.parquet"))]
    real_files = [p for p in all_files if p.name.startswith("WELL-") and " (" not in p.stem]
    inventory = pd.DataFrame({
        "relative_path": [str(p.relative_to(source)) for p in real_files],
        "entity_id": [p.name.split("_", 1)[0] for p in real_files],
        "event_code": [int(p.parent.name) for p in real_files],
        "episode_id": [p.stem for p in real_files],
        "rows": [pq.ParquetFile(p).metadata.num_rows for p in real_files],
    })
    inventory["selection_key"] = inventory["relative_path"].map(
        lambda value: hashlib.sha256(value.encode()).hexdigest()
    )
    transient_offset = parser.getint("EVENTS", "TRANSIENT_OFFSET", fallback=100)
    return parser.get("VERSION", "DATASET"), len(all_files), inventory, transient_offset


def select_population(inventory, files_per_pair):
    """Select up to n deterministic recordings per (well, event class)."""

    return (
        inventory.sort_values("selection_key")
        .groupby(["entity_id", "event_code"], as_index=False)
        .head(files_per_pair)
        .sort_values(["entity_id", "event_code", "relative_path"])
        .reset_index(drop=True)
    )


def add_label_evidence(source, selection, transient_offset):
    """Read actual class values used to audit and constrain the split."""

    evidence = []
    for item in selection.itertuples(index=False):
        labels = pd.to_numeric(
            pd.read_parquet(Path(source) / item.relative_path, columns=["class"])["class"],
            errors="coerce",
        )
        raw_codes = set(labels.dropna().astype(int))
        fault_codes = tuple(sorted(
            code - transient_offset if code >= transient_offset else code
            for code in raw_codes if code != 0
        ))
        evidence.append({
            "episode_id": item.episode_id,
            "has_normal": 0 in raw_codes,
            "has_fault": bool(fault_codes),
            "observed_fault_codes": fault_codes,
            "directory_label_confirmed": (
                (item.event_code == 0 and 0 in raw_codes)
                or (item.event_code != 0 and item.event_code in fault_codes)
            ),
        })
    return selection.merge(pd.DataFrame(evidence), on="episode_id", validate="one_to_one")


def make_well_partitions(selection, seed, shares=(0.35, 0.40)):
    """Create a fixed 35/40/25 whole-well split without searching labels."""

    wells = sorted(selection["entity_id"].unique())
    n_calibration = round(len(wells) * shares[0])
    n_development = round(len(wells) * shares[1])
    if min(n_calibration, n_development, len(wells) - n_calibration - n_development) < 1:
        raise ValueError(f"{len(wells)} wells cannot fill three partitions")

    # Keep the established hash ordering so the ten holdout wells do not change.
    ordered = sorted(wells, key=lambda well: hashlib.sha256(
        f"threew_split_v3|{seed}|{well}".encode()).hexdigest())
    return {
        well: ("calibration" if i < n_calibration
               else "development" if i < n_calibration + n_development
               else "holdout")
        for i, well in enumerate(ordered)
    }


version, all_file_count, inventory, transient_offset = read_inventory(SOURCE)
selected = select_population(inventory, FILES_PER_WELL_EVENT)
one_per_pair = set(select_population(inventory, 1)["relative_path"])
assert one_per_pair <= set(selected["relative_path"]), "Selection must be nested"
selected = add_label_evidence(SOURCE, selected, transient_offset)
assignment = make_well_partitions(selected, SPLIT_SEED)
selected["partition"] = selected["entity_id"].map(assignment)

entity_partitions = pd.DataFrame({
    "entity_id": sorted(assignment),
    "partition": [assignment[well] for well in sorted(assignment)],
    "split_version": f"threew_well_hash_v4_35_40_25_seed{SPLIT_SEED}",
})[SPLIT_SCHEMAS["entity_partitions"]]

split_coverage = selected.groupby("partition").agg(
    recordings=("episode_id", "size"),
    wells=("entity_id", "nunique"),
    recordings_with_normal=("has_normal", "sum"),
    recordings_with_fault=("has_fault", "sum"),
)
split_coverage["one_fault_recall_step"] = (
    1 / split_coverage["recordings_with_fault"].replace(0, np.nan)
)
split_coverage["episodes_per_well"] = (
    split_coverage["recordings"] / split_coverage["wells"]
)
exploded_faults = selected.explode("observed_fault_codes")
actual_fault_coverage = pd.crosstab(
    exploded_faults["partition"], exploded_faults["observed_fault_codes"],
)
label_mismatches = selected.loc[
    ~selected["directory_label_confirmed"],
    ["relative_path", "event_code", "has_normal", "observed_fault_codes"],
]
display(pd.Series({
    "dataset_version": version,
    "files_in_source": all_file_count,
    "real_well_files": len(inventory),
    "selected_recordings": len(selected),
    "selected_wells": selected["entity_id"].nunique(),
    "selected_rows": int(selected["rows"].sum()),
    "split_seed": SPLIT_SEED,
}, name="value").to_frame())
display(split_coverage)
display(actual_fault_coverage)
print("Recordings from the same well are correlated; report wells and recordings together.")
if not label_mismatches.empty:
    print("Directory labels not confirmed by class values (reported, not relabelled):")
    display(label_mismatches)

assert version == "2.0.0"
assert all_file_count == EXPECTED_FILES, f"Inventory is {all_file_count}, expected {EXPECTED_FILES}"
assert entity_partitions.groupby("entity_id")["partition"].nunique().max() == 1
assert set(entity_partitions["partition"]) == {"calibration", "development", "holdout"}
assert split_coverage.loc[["development", "holdout"], "recordings_with_normal"].gt(0).all()
assert split_coverage.loc[["development", "holdout"], "recordings_with_fault"].gt(0).all()

In [ ]:
def sentinel_audit(source, selection):
    """Count only the exact sentinel values declared above."""

    rows = []
    native_by_metric = metric_map.set_index("metric_id")["native_field"]
    for metric_id, rule in SOURCE_QUALITY_RULES.items():
        for value in rule.get("invalid_values", ()):
            native_field = native_by_metric.loc[metric_id]
            count = 0
            for relative_path in selection["relative_path"]:
                path = Path(source) / relative_path
                if native_field in pq.ParquetFile(path).schema_arrow.names:
                    values = pd.read_parquet(path, columns=[native_field])[native_field]
                    count += int(values.eq(value).sum())
            rows.append({
                "metric_id": metric_id,
                "sentinel_value": value,
                "rows_marked_invalid": count,
                "provenance": rule["source"],
            })
    return pd.DataFrame(rows)


sentinel_report = sentinel_audit(SOURCE, selected)
display(sentinel_report)
assert sentinel_report["rows_marked_invalid"].gt(0).all()


## 4. Translate

One recording is read at a time. A metric with no value anywhere in a
recording is **absent** from that episode; a partial null inside an observed
metric is an `invalid` observation. That is the same availability rule the
Telecom pack now applies, so `valid_rate` and coverage mean the same thing in
both sectors.

`observable_ts` and `impact_ts` come from the published `1xx` transient and
`x` steady class phases. Their `label_source` says so: they are labelling
artefacts, not independent physical or business-impact timestamps.

In [ ]:
def read_recording(path):
    """Read one recording and validate its time axis."""

    frame = pd.read_parquet(path)
    if frame.empty:
        raise ValueError(f"Empty 3W recording: {path}")
    if "timestamp" in frame.columns:
        frame = frame.rename(columns={"timestamp": "event_ts"})
    elif isinstance(frame.index, pd.DatetimeIndex):
        frame = frame.reset_index().rename(columns={frame.index.name or "index": "event_ts"})
    else:
        raise ValueError(f"No timestamp column or DatetimeIndex in {path}")
    frame = frame.reset_index(drop=True)
    frame["event_ts"] = pd.to_datetime(frame["event_ts"], utc=True, errors="raise")
    if frame["event_ts"].isna().any():
        raise ValueError(f"Missing timestamps in {path}")
    if not frame["event_ts"].is_monotonic_increasing:
        raise ValueError(f"Timestamps are not ordered in {path}")
    if frame["event_ts"].duplicated().any():
        raise ValueError(f"Duplicate timestamps in {path}")
    return frame


def runs(series):
    """Yield (start, end_exclusive, value) for each contiguous run."""

    if series.empty:
        return
    blocks = (series.ne(series.shift()) & ~(series.isna() & series.shift().isna())).cumsum()
    for _, index in series.groupby(blocks).groups.items():
        yield int(index[0]), int(index[-1]) + 1, series.iloc[int(index[0])]


def telemetry_batches(source, selection, batch_rows):
    """Yield long telemetry, one recording at a time."""

    rename = dict(zip(metric_map["native_field"], metric_map["metric_id"]))
    for item in selection.itertuples(index=False):
        native = read_recording(Path(source) / item.relative_path)
        present = [f for f in metric_map["native_field"]
                   if f in native.columns and native[f].notna().any()]
        if not present:
            raise ValueError(f"No observed metrics in {item.relative_path}")
        for start in range(0, len(native), batch_rows):
            chunk = native.iloc[start:start + batch_rows]
            wide = chunk[["event_ts", *present]].rename(columns=rename)
            long = wide.melt(id_vars="event_ts", var_name="metric_id", value_name="value")
            long["entity_id"] = item.entity_id
            long["episode_id"] = item.episode_id
            numeric = pd.to_numeric(long["value"], errors="coerce")
            invalid = numeric.isna() | ~np.isfinite(numeric)
            for metric_id, rule in SOURCE_QUALITY_RULES.items():
                rows = long["metric_id"].eq(metric_id)
                if "allowed_values" in rule:
                    invalid |= rows & ~numeric.isin(rule["allowed_values"])
                if "minimum" in rule:
                    invalid |= rows & numeric.lt(rule["minimum"])
                if "maximum" in rule:
                    invalid |= rows & numeric.gt(rule["maximum"])
                if "invalid_values" in rule:
                    invalid |= rows & numeric.isin(rule["invalid_values"])
            long["value"] = numeric
            long["quality_code"] = np.where(invalid, "invalid", "measured")
            yield long


def event_definitions(source):
    parser = configparser.ConfigParser(interpolation=None)
    parser.read(Path(source) / "dataset.ini", encoding="utf-8")
    names = [n.strip() for n in parser.get("EVENTS", "NAMES").replace("\n", "").split(",") if n.strip()]
    definitions = {
        parser.getint(name, "LABEL"): {
            "description": parser.get(name, "DESCRIPTION"),
            "has_transient": parser.getboolean(
                name, "TRANSIENT", fallback=parser.getint(name, "LABEL") != 0
            ),
        }
        for name in names
    }
    if len(definitions) != len(names):
        raise ValueError("dataset.ini contains duplicate event labels")
    return definitions, parser.getint("EVENTS", "TRANSIENT_OFFSET", fallback=100)


def recording_truth(frame, item, definitions, transient_offset):
    """Convert one recording's class and state columns into evaluation rows."""

    missing = {"event_ts", *EVALUATION_ONLY_FIELDS} - set(frame.columns)
    if missing:
        raise ValueError(f"Evaluation labels missing from {item.relative_path}: {sorted(missing)}")

    timestamps = frame["event_ts"].reset_index(drop=True)
    final_end = timestamps.iloc[-1] + pd.Timedelta(seconds=CADENCE_SECONDS)
    labels = pd.to_numeric(frame["class"], errors="raise").reset_index(drop=True)
    allowed = set(definitions) | {
        transient_offset + code for code, d in definitions.items() if d["has_transient"]
    }
    unexpected = set(labels.dropna().astype(int)) - allowed
    if unexpected:
        raise ValueError(f"Unexpected class labels in {item.relative_path}: {sorted(unexpected)}")

    def end_of(index):
        return timestamps.iloc[index] if index < len(timestamps) else final_end

    conditions = [
        {"entity_id": item.entity_id, "start_ts": timestamps.iloc[start],
         "end_ts": end_of(end), "condition_code": str(int(value)) if float(value).is_integer() else str(value),
         "label_source": "3w_state_label", "source_instance_id": item.episode_id}
        for start, end, value in runs(pd.to_numeric(frame["state"], errors="coerce").reset_index(drop=True))
        if pd.notna(value)
    ]

    active = labels.map(lambda v: np.nan if pd.isna(v) or int(v) == 0
                        else float(int(v) - transient_offset) if int(v) >= transient_offset
                        else float(int(v))).astype(float)
    events, intervals = [], []
    for number, (start, end, code) in enumerate(r for r in runs(active) if pd.notna(r[2])):
        code = int(code)
        segment = labels.iloc[start:end]
        transient = segment.index[segment.eq(transient_offset + code)]
        steady = segment.index[segment.eq(code)]
        if len(transient) and len(steady) and steady[0] < transient[0]:
            raise ValueError(f"Steady phase precedes transient phase in {item.relative_path}")
        fault_id = f"3W-{item.episode_id}-{code}-{number:02d}"
        events.append({
            "fault_id": fault_id, "fault_type": definitions[code]["description"],
            "domain_id": item.entity_id, "onset_ts": timestamps.iloc[start],
            "observable_ts": timestamps.iloc[int(transient[0])] if len(transient) else timestamps.iloc[start],
            "impact_ts": timestamps.iloc[int(steady[0])] if len(steady) else pd.NaT,
            "end_ts": end_of(end), "group_id": None,
            "label_source": "3w_class_phase_derived", "source_instance_id": item.episode_id,
        })
        intervals.append({
            "fault_id": fault_id, "entity_id": item.entity_id,
            "start_ts": timestamps.iloc[start], "end_ts": end_of(end),
            "label_source": "3w_class_interval", "source_instance_id": item.episode_id,
        })
    return events, intervals, conditions


def evaluation_tables(source, selection):
    definitions, transient_offset = event_definitions(source)
    events, intervals, conditions = [], [], []
    for item in selection.itertuples(index=False):
        frame = read_recording(Path(source) / item.relative_path)
        recording_events, recording_intervals, recording_conditions = recording_truth(
            frame, item, definitions, transient_offset
        )
        events += recording_events
        intervals += recording_intervals
        conditions += recording_conditions
    return {
        name: pd.DataFrame(rows, columns=EVAL_SCHEMAS[name])
        for name, rows in (("fault_events", events),
                           ("fault_entity_intervals", intervals),
                           ("condition_states", conditions))
    }

In [ ]:
def observed_catalogue(source, selection):
    """Keep metrics with at least one value in the selected population."""

    remaining = set(metric_map["native_field"])
    observed = set()
    for relative_path in selection["relative_path"]:
        parquet = pq.ParquetFile(Path(source) / relative_path)
        names = parquet.schema_arrow.names
        candidates = sorted(remaining.intersection(names))
        if candidates:
            values = pd.read_parquet(Path(source) / relative_path, columns=candidates)
            observed.update(values.columns[values.notna().any()])
            remaining -= observed
        if not remaining:
            break
    return metric_map.loc[metric_map["native_field"].isin(observed)]


def build_threew_pack(source, destination, selection, partitions, *,
                      include_evaluation=True, batch_rows=BATCH_ROWS):
    source, destination = Path(source), Path(destination)
    catalogue = observed_catalogue(source, selection)
    if catalogue.empty:
        raise ValueError("No mapped measurement was observed in the selection")

    return save_pack(
        destination,
        sector="petrobras_3w",
        pack_version="0.8.0",
        source_info={
            "source_id": "petrobras-3w-2.0.0",
            "source_root": str(source),
            "selection_rule": f"lowest {FILES_PER_WELL_EVENT} SHA-256 path keys per real well and event directory",
            "split_rule": f"fixed whole-well 35/40/25 hash split, seed {SPLIT_SEED}; labels never searched",
            "fields_routed_only_to_evaluation": EVALUATION_ONLY_FIELDS,
            "source_quality_rules": SOURCE_QUALITY_RULES,
            "files": [
                source_file(source / "dataset.ini", source, "source_metadata"),
                *[source_file(source / p, source, "model_and_evaluation_source")
                  for p in selection["relative_path"]],
            ],
        },
        telemetry=telemetry_batches(source, selection, batch_rows),
        catalogue=catalogue,
        entities=pd.DataFrame({
            "entity_id": sorted(selection["entity_id"].unique()), "entity_type": "oil_well",
        }),
        episodes=selection[["episode_id", "entity_id"]]
            .drop_duplicates().assign(episode_basis="source_recording"),
        splits={"entity_partitions": partitions},
        evaluation=evaluation_tables(source, selection) if include_evaluation else {},
        notes=[
            f"Real WELL recordings only; up to {FILES_PER_WELL_EVENT} deterministic files per well and event directory.",
            "Every source recording is a distinct episode; no obligation between recordings.",
            "A metric with no value in a recording is absent, not a run of invalid rows.",
            "Gaps are found inside a recording only, never between recordings.",
            "Naive source timestamps are read as UTC; no physical source timezone is inferred.",
            "Development and holdout each contain recordings with actual normal and fault class values.",
            "Folder/class mismatches are reported; splits never use folder names as label evidence.",
            "Dataset.ini constraints and three verified historian sentinels mark finite values invalid.",
        ],
    )


if RUN_BUILD:
    pack_manifest = build_threew_pack(SOURCE, PACK_ROOT, selected, entity_partitions)
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

sentinel_report.to_csv(PACK_ROOT / "sentinel_audit.csv", index=False)
split_coverage.reset_index().to_csv(
    PACK_ROOT / "population_report.csv", index=False
)
actual_fault_coverage.reset_index().to_csv(
    PACK_ROOT / "class_coverage_report.csv", index=False
)
display(pd.Series(pack_manifest["core_row_counts"], name="rows").to_frame())

## 5. Truth-isolation test

The expanded population is not rebuilt. A small representative subset is
translated before and after `class` and `state` are removed. `PACK-CORE` must
be identical, a deliberately leaky class signature must change, and asking
for evaluation from the redacted source must fail explicitly.

In [ ]:
subset = pd.concat([
    selected.loc[selected["event_code"].eq(0)].head(1),
    selected.loc[selected["event_code"].gt(0)].drop_duplicates("event_code").head(2),
], ignore_index=True)
subset_partitions = entity_partitions.loc[
    entity_partitions["entity_id"].isin(subset["entity_id"])
]


def make_redacted_source(source, destination, selection):
    destination.mkdir(parents=True)
    shutil.copy2(Path(source) / "dataset.ini", destination / "dataset.ini")
    for relative_path in selection["relative_path"]:
        target = destination / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        frame = pd.read_parquet(Path(source) / relative_path)
        redacted = frame.drop(columns=EVALUATION_ONLY_FIELDS)
        # Official 3W files store time in a DatetimeIndex. Preserve it.
        redacted.to_parquet(target)


def leaky_signature(source, selection):
    values = []
    for relative_path in selection["relative_path"]:
        path = Path(source) / relative_path
        if "class" not in pq.ParquetFile(path).schema_arrow.names:
            return "missing"
        values.append(pd.read_parquet(path, columns=["class"]))
    return str(pd.concat(values)["class"].value_counts().sort_index().to_dict())


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    redacted = temporary / "native_redacted"
    make_redacted_source(SOURCE, redacted, subset)

    build_threew_pack(SOURCE, temporary / "pack_original", subset, subset_partitions,
                      include_evaluation=True)
    build_threew_pack(redacted, temporary / "pack_redacted", subset, subset_partitions,
                      include_evaluation=False)

    assert pack_fingerprint(temporary / "pack_original") == pack_fingerprint(temporary / "pack_redacted")
    assert leaky_signature(SOURCE, subset) != leaky_signature(redacted, subset)

    try:
        build_threew_pack(redacted, temporary / "pack_needs_truth", subset, subset_partitions,
                          include_evaluation=True)
    except ValueError as error:
        assert "Evaluation labels missing" in str(error)
    else:
        raise AssertionError("Evaluation build accepted a source without labels")

print("PASS — PACK-CORE is unchanged after class and state are removed")
print("PASS — the negative control detects the removed labels")
print("PASS — evaluation cannot be requested when labels are absent")

## 6. Contract fit and evidence for the next stage

What the source could not express is evidence about the contract, not a
failure to hide behind fabricated data. Fault counts per partition say
whether a per-type detection rate is estimable at all.

In [ ]:
display(pd.Series({
    "represented_cleanly": [
        "multivariate telemetry within recordings",
        "real well identity and source recording identity",
        "metric-level observation presence",
        "never-observed metrics distinguished from attempted invalid values",
        "gaps inside a recording, undefined between recordings",
        "class labels physically isolated in PACK-EVAL",
        "whole-well development partitions",
    ],
    "not_expressible_without_invention": [
        "continuous service obligation between recordings",
        "shared manifold topology",
        "cross-well cause groups",
        "operator tickets or maintenance actions",
        "independent business-impact timestamps",
    ],
}, name="finding").to_frame())

display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "metric_catalogue.parquet"))
display(pd.read_parquet(PACK_ROOT / "SPLITS" / "entity_partitions.parquet")
        .groupby("partition").size().rename("wells").to_frame())

coverage = fault_coverage(PACK_ROOT)
display(coverage)
thin = coverage.loc[
    coverage["partition"].eq("holdout")
    & coverage["scoreable_faults"].lt(5)
]
if not thin.empty:
    print("WARNING — holdout fault types with fewer than five faults:")
    display(thin)
if coverage["unscoreable_faults"].sum():
    print("WARNING — some declared faults do not overlap observable telemetry.")
if coverage["cross_partition_faults"].sum():
    print("WARNING — some faults span more than one entity partition.")

print("Pack root:", PACK_ROOT)
print("Next: 01B_COMMON_CANONICAL_ADAPTER.ipynb")